In [32]:
import pandas as pd
import numpy as np
import re
import pymorphy3
from nltk.corpus import stopwords
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import pickle
import os
from datetime import datetime

In [2]:
nltk.download('stopwords', quiet=True)

True

In [20]:
# Загружаем тренировочный датасет
train_df = pd.read_csv('../data/service_desk_train.csv', sep=';', encoding='utf-8')

In [21]:
# Загружаем тестовый датасет
test_df = pd.read_csv('../data/service_desk_test.csv', sep=';', encoding='utf-8')

## 01 Класс для предобработки текста ##

In [22]:
class TextPreprocessor:
    """Класс для очистки и предобработки русских текстов"""
    
    def __init__(self):
        self.morph = pymorphy3.MorphAnalyzer()
        self.russian_stopwords = set(stopwords.words('russian'))
        print("✓ TextPreprocessor инициализирован")
    
    def clean_text(self, text):
        """Очистка текста от спецсимволов"""
        if pd.isna(text):
            return ""
        
        # Преобразуем в строку и делаем lowercase
        text = str(text).lower()
        
        # Удаляем email
        text = re.sub(r'\S+@\S+', ' ', text)
        
        # Удаляем URL
        text = re.sub(r'http\S+|www\S+', ' ', text)
        
        # Удаляем даты (форматы: 12.02.2025, 12/02/2025, 12-02-2025)
        text = re.sub(r'\d{1,2}[./\-]\d{1,2}[./\-]\d{2,4}', ' DATE ', text)
        
        # Заменяем числа на токен NUM
        text = re.sub(r'\d+', ' NUM ', text)
        
        # Удаляем специальные символы, оставляем только буквы, цифры и пробелы
        text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
        
        # Удаляем множественные пробелы
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()
    
    def lemmatize(self, text):
        """Лемматизация (преобразование слов в начальную форму)"""
        if not text:
            return ""
        
        words = text.split()
        lemmatized_words = []
        
        for word in words:
            # Пропускаем стоп-слова и очень короткие слова
            if word not in self.russian_stopwords and len(word) > 2:
                # Получаем нормальную форму слова
                parsed = self.morph.parse(word)[0]
                lemma = parsed.normal_form
                lemmatized_words.append(lemma)
        
        return ' '.join(lemmatized_words)
    
    def preprocess(self, text):
        """Полная предобработка текста"""
        text = self.clean_text(text)
        text = self.lemmatize(text)
        return text

In [23]:
# Тестируем класс
preprocessor = TextPreprocessor()

print("\n" + "="*80)
print("🧪 ТЕСТИРОВАНИЕ PREPROCESSOR")
print("="*80)

example_text = train_df['Тема'].iloc[0]
print(f"\n📝 Исходный текст:\n   {example_text}")

cleaned = preprocessor.clean_text(example_text)
print(f"\n🔧 После очистки:\n   {cleaned}")

lemmatized = preprocessor.lemmatize(cleaned)
print(f"\n📎 После лемматизации:\n   {lemmatized}")

✓ TextPreprocessor инициализирован

🧪 ТЕСТИРОВАНИЕ PREPROCESSOR

📝 Исходный текст:
   Утерян пропуск Рымарь В.С.

🔧 После очистки:
   утерян пропуск рымарь в с

📎 После лемматизации:
   утерять пропуск рымарь


## 02 Подготовка данных ##

In [24]:
# ИСПОЛЬЗУЕМ ТОЛЬКО КОЛОНКУ "Тема" (БЕЗ "Активности")
# НЕ используем "Пользовательское поле (Активность)" - DATA LEAKAGE!
print("\n" + "="*80)
print("📝 ПОДГОТОВКА ТЕКСТОВЫХ ДАННЫХ")
print("="*80)

print("\n🎯 Стратегия обработки:")
print("   ✓ Используем: Колонка 'Тема'")
print("   ✗ НЕ используем: 'Пользовательское поле (Активность)' (нет в test.csv)")
print("   ✗ НЕ используем: 'Срок исполнения' (100% пропусков)")

# Объединяем Тема (в нашем случае - только Тема)
train_df['full_text'] = train_df['Тема'].fillna('')

print(f"\n📊 Статистика текстов:")
print(f"   Записей с текстом: {(train_df['full_text'] != '').sum()}")
print(f"   Записей без текста: {(train_df['full_text'] == '').sum()}")
print(f"   Средняя длина текста: {train_df['full_text'].str.len().mean():.0f} символов")


📝 ПОДГОТОВКА ТЕКСТОВЫХ ДАННЫХ

🎯 Стратегия обработки:
   ✓ Используем: Колонка 'Тема'
   ✗ НЕ используем: 'Пользовательское поле (Активность)' (нет в test.csv)
   ✗ НЕ используем: 'Срок исполнения' (100% пропусков)

📊 Статистика текстов:
   Записей с текстом: 8565
   Записей без текста: 0
   Средняя длина текста: 46 символов


## 03 Проведение предобработки всего текста ##

In [25]:
print("\n" + "="*80)
print("🔧 ПРЕДОБРАБОТКА ВСЕХ ТЕКСТОВ")
print("="*80)

# Применяем предобработку
print("\nОбработка train текстов...")
train_df['processed_text'] = train_df['full_text'].apply(preprocessor.preprocess)

# Для test датасета (если нужно, но целевой переменной нет)
print("Обработка test текстов...")
test_df['full_text'] = test_df['Тема'].fillna('')
test_df['processed_text'] = test_df['full_text'].apply(preprocessor.preprocess)

print("\n✓ Предобработка завершена!")

# Статистика
print(f"\n📊 Статистика обработанных текстов:")
print(f"   Train - средняя длина после обработки: {train_df['processed_text'].str.len().mean():.0f} символов")
print(f"   Test - средняя длина после обработки: {test_df['processed_text'].str.len().mean():.0f} символов")

# Примеры
print(f"\n{'='*80}")
print("✅ ПРИМЕРЫ ПОСЛЕ ПРЕДОБРАБОТКИ")
print(f"{'='*80}")

for i in range(3):
    print(f"\n📌 Пример {i+1}:")
    print(f"  Исходный: {train_df['full_text'].iloc[i][:80]}")
    print(f"  Обработанный: {train_df['processed_text'].iloc[i][:80]}")


🔧 ПРЕДОБРАБОТКА ВСЕХ ТЕКСТОВ

Обработка train текстов...
Обработка test текстов...

✓ Предобработка завершена!

📊 Статистика обработанных текстов:
   Train - средняя длина после обработки: 39 символов
   Test - средняя длина после обработки: 36 символов

✅ ПРИМЕРЫ ПОСЛЕ ПРЕДОБРАБОТКИ

📌 Пример 1:
  Исходный: Утерян пропуск Рымарь В.С.
  Обработанный: утерять пропуск рымарь

📌 Пример 2:
  Исходный: Доступ к ресурсу БФТ
  Обработанный: доступ ресурс бфт

📌 Пример 3:
  Исходный: Гостевой пропуск для Ермачкова Виктория Игоревна
  Обработанный: гостевой пропуск ермачкова виктория игоревич


## 04 Кодирование целевой переменной ##

In [26]:
print("\n" + "="*80)
print("🏷️ КОДИРОВАНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("="*80)

# Label Encoding для целевой переменной
label_encoder = LabelEncoder()
train_df['target_encoded'] = label_encoder.fit_transform(train_df['Тип задачи'])

print(f"\n✓ Label encoder создан")
print(f"   Количество классов: {len(label_encoder.classes_)}")

# Показываем соответствие
print(f"\n📊 Соответствие категорий и кодов:")
print(f"{'='*80}")

correspondence = pd.DataFrame({
    'Код': range(len(label_encoder.classes_)),
    'Категория': label_encoder.classes_,
    'Количество': [
        (train_df['target_encoded'] == i).sum() 
        for i in range(len(label_encoder.classes_))
    ]
})

correspondence = correspondence.sort_values('Количество', ascending=False)
print(correspondence.to_string(index=False))

print(f"\n{'='*80}")


🏷️ КОДИРОВАНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ

✓ Label encoder создан
   Количество классов: 17

📊 Соответствие категорий и кодов:
 Код                     Категория  Количество
   6        Запрос на обслуживание        4739
  14          Подзадача увольнение        1562
  12                     Подзадача        1192
   5      Заказ гостевого пропуска         228
   1                     HR: Приём         180
   8    Заявка на выход сотрудника         165
   7  Заявка на билет и проживание         152
  13   Подзадача основные средства         123
   3                HR: Увольнение          85
  11      Изменение условий работы          38
   9     Заявка на согласование ВМ          32
   2    HR: Техническое увольнение          21
   0  HR: Перевод через увольнение          15
  16         Уведомление о работах          13
  10 Изменение персональных данных          11
  15              Согласование VDI           5
   4                 Заказ визиток           4



**Вывод**  
Имеется очень мало примеров для категорий с "HR: Перевод через увольнение" по "Заказ визиток". Здесь могут быть ошибки у обученной модели

## 05 TF-IDF векторизация ##

In [27]:
print("\n" + "="*80)
print("🔢 TF-IDF ВЕКТОРИЗАЦИЯ")
print("="*80)

# Параметры TF-IDF
print("\n📋 Параметры TF-IDF:")
print("   max_features: 8000 (максимальное количество признаков)")
print("   ngram_range: (1, 2) (1-граммы и 2-граммы)")
print("   min_df: 2 (слово должно быть в минимум 2 документах)")
print("   max_df: 0.9 (слово может быть максимум в 90% документов)")

# Создаем и обучаем TF-IDF
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

print("\n🔨 Обучение TF-IDF на train текстах...")
X_train_tfidf = vectorizer.fit_transform(train_df['processed_text'])

print("✓ Готово!")

print(f"\n📊 Результаты TF-IDF векторизации:")
print(f"   Количество текстов: {X_train_tfidf.shape[0]}")
print(f"   Количество признаков (слов/фраз): {X_train_tfidf.shape[1]}")
print(f"   Тип матрицы: {type(X_train_tfidf).__name__} (разреженная)")
print(f"   Плотность: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100:.2f}%")

# Информация о топ признаков
feature_names = vectorizer.get_feature_names_out()
print(f"\n📝 Примеры признаков (первые 30):")
print(f"   {', '.join(feature_names[:30])}")

print(f"\n🔝 Примеры признаков (последние 30):")
print(f"   {', '.join(feature_names[-30:])}")


🔢 TF-IDF ВЕКТОРИЗАЦИЯ

📋 Параметры TF-IDF:
   max_features: 8000 (максимальное количество признаков)
   ngram_range: (1, 2) (1-граммы и 2-граммы)
   min_df: 2 (слово должно быть в минимум 2 документах)
   max_df: 0.9 (слово может быть максимум в 90% документов)

🔨 Обучение TF-IDF на train текстах...
✓ Готово!

📊 Результаты TF-IDF векторизации:
   Количество текстов: 8565
   Количество признаков (слов/фраз): 4402
   Тип матрицы: csr_matrix (разреженная)
   Плотность: 0.17%

📝 Примеры признаков (первые 30):
   ads, ads bftcom, agile, aisubp, alarm, alarms, altova, amur, any, anyconnect, api, api jira, aquarius, attention, attention batch, aug, auth, auth bot, authenticator, avanpost, avanpost authenticator, azk, bamboo, batch, batch alarms, belgorod, belgorod kbr, bft, bft auth, bft kiselev

🔝 Примеры признаков (последние 30):
   юлия святославович, юлия эдуардович, юлия юриевич, юнеть, юнеть дмитрий, юридический, юридический управление, юриевич, юриевич бфт, юриевич левитин, юриевич мо

## 06 Подготовка данных для обучения модели ##

In [28]:
print("\n" + "="*80)
print("📊 ФИНАЛЬНАЯ ПОДГОТОВКА ДАННЫХ")
print("="*80)

# Целевая переменная
y_train = train_df['target_encoded'].values

# Признаки (уже векторизованы)
X_train = X_train_tfidf

print(f"\n✓ Данные готовы для обучения модели:")
print(f"   X_train форма: {X_train.shape}")
print(f"   y_train форма: {y_train.shape}")
print(f"   Типы данных: X_train = {type(X_train).__name__}, y_train = {type(y_train).__name__}")

# Проверка на пропуски
print(f"\n🔍 Проверка на пропуски:")
print(f"   Пропусков в X_train: 0 (разреженная матрица)")
print(f"   Пропусков в y_train: {np.isnan(y_train).sum()}")


📊 ФИНАЛЬНАЯ ПОДГОТОВКА ДАННЫХ

✓ Данные готовы для обучения модели:
   X_train форма: (8565, 4402)
   y_train форма: (8565,)
   Типы данных: X_train = csr_matrix, y_train = ndarray

🔍 Проверка на пропуски:
   Пропусков в X_train: 0 (разреженная матрица)
   Пропусков в y_train: 0


## 07 Сохранение артефактов ##

In [33]:
print("\n" + "="*80)
print("💾 СОХРАНЕНИЕ АРТЕФАКТОВ")
print("="*80)

# Создаем директорию если не существует
os.makedirs('../models/v1.0', exist_ok=True)

# Сохраняем preprocessor
print("\n1. Сохранение preprocessor...")
with open('../models/v1.0/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("   ✓ ../models/v1.0/preprocessor.pkl")

# Сохраняем vectorizer
print("\n2. Сохранение vectorizer...")
with open('../models/v1.0/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("   ✓ ../models/v1.0/vectorizer.pkl")

# Сохраняем label encoder
print("\n3. Сохранение label encoder...")
with open('../models/v1.0/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("   ✓ ../models/v1.0/label_encoder.pkl")

# Сохраняем обработанный датасет
print("\n4. Сохранение обработанного датасета...")
train_df_save = train_df[['id', 'Тема', 'Тип задачи', 'full_text', 'processed_text', 'target_encoded']].copy()
train_df_save.to_csv('../data/processed/train_processed.csv', sep=';', encoding='utf-8', index=False)
print("   ✓ ../data/processed/train_processed.csv")

# Сохраняем метаданные
print("\n5. Сохранение метаданных...")
metadata = {
    'preprocessing_date': datetime.now().isoformat(),
    'num_classes': len(label_encoder.classes_),
    'classes': label_encoder.classes_.tolist(),
    'num_train_samples': len(train_df),
    'num_features': X_train.shape[1],
    'vectorizer_params': {
        'max_features': 8000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.9
    },
    'strategy': 'Only Tema column (no Activity column due to data leakage)'
}

import json
with open('../models/v1.0/preprocessing_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("   ✓ ../models/v1.0/preprocessing_metadata.json")

print("\n" + "="*80)
print("✅ ВСЕ АРТЕФАКТЫ СОХРАНЕНЫ УСПЕШНО!")
print("="*80)

print("\n📦 Сохраненные файлы:")
print("   - preprocessor.pkl (текстовый процессор)")
print("   - vectorizer.pkl (TF-IDF векторизатор)")
print("   - label_encoder.pkl (кодировщик категорий)")
print("   - preprocessing_metadata.json (метаданные)")
print("   - train_processed.csv (обработанные данные)")


💾 СОХРАНЕНИЕ АРТЕФАКТОВ

1. Сохранение preprocessor...
   ✓ ../models/v1.0/preprocessor.pkl

2. Сохранение vectorizer...
   ✓ ../models/v1.0/vectorizer.pkl

3. Сохранение label encoder...
   ✓ ../models/v1.0/label_encoder.pkl

4. Сохранение обработанного датасета...
   ✓ ../data/processed/train_processed.csv

5. Сохранение метаданных...
   ✓ ../models/v1.0/preprocessing_metadata.json

✅ ВСЕ АРТЕФАКТЫ СОХРАНЕНЫ УСПЕШНО!

📦 Сохраненные файлы:
   - preprocessor.pkl (текстовый процессор)
   - vectorizer.pkl (TF-IDF векторизатор)
   - label_encoder.pkl (кодировщик категорий)
   - preprocessing_metadata.json (метаданные)
   - train_processed.csv (обработанные данные)


## 08 Финальная проверка и статистика ##

In [35]:
print("\n" + "="*80)
print("📈 ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*80)

print(f"\n✓ Обучение завершено успешно!")

print(f"\n📊 Статистика датасета:")
print(f"   Всего примеров: {len(train_df)}")
print(f"   Уникальных категорий: {len(label_encoder.classes_)}")
print(f"   Количество признаков (слов/фраз): {X_train.shape[1]}")

print(f"\n📝 Статистика по категориям:")
for code, category in enumerate(label_encoder.classes_):
    count = (y_train == code).sum()
    percentage = (count / len(y_train)) * 100
    print(f"   {code:2d}. {category:40s} - {count:5d} ({percentage:5.1f}%)")

print(f"\n🔧 Статистика текстов:")
print(f"   Средняя длина исходного текста: {train_df['full_text'].str.len().mean():.0f} символов")
print(f"   Средняя длина после предобработки: {train_df['processed_text'].str.len().mean():.0f} символов")
print(f"   Средняя длина слова: {train_df['processed_text'].str.len().mean() / train_df['processed_text'].str.split().str.len().mean():.1f} символов")


📈 ФИНАЛЬНАЯ СТАТИСТИКА

✓ Обучение завершено успешно!

📊 Статистика датасета:
   Всего примеров: 8565
   Уникальных категорий: 17
   Количество признаков (слов/фраз): 4402

📝 Статистика по категориям:
    0. HR: Перевод через увольнение             -    15 (  0.2%)
    1. HR: Приём                                -   180 (  2.1%)
    2. HR: Техническое увольнение               -    21 (  0.2%)
    3. HR: Увольнение                           -    85 (  1.0%)
    4. Заказ визиток                            -     4 (  0.0%)
    5. Заказ гостевого пропуска                 -   228 (  2.7%)
    6. Запрос на обслуживание                   -  4739 ( 55.3%)
    7. Заявка на билет и проживание             -   152 (  1.8%)
    8. Заявка на выход сотрудника               -   165 (  1.9%)
    9. Заявка на согласование ВМ                -    32 (  0.4%)
   10. Изменение персональных данных            -    11 (  0.1%)
   11. Изменение условий работы                 -    38 (  0.4%)
   12. Подзадача  

**Замечания**
1. **Data Leakage:** используется ТОЛЬКО колонка "Тема"
3. **Все артефакты сохранены:** готовы к использованию в production
4. **Метаданные сохранены:** для воспроизводимости